# 10.11 能反過來用描述找圖片嗎？


「這張圖對應哪段文字」與「這段文字對應哪張圖」看似相同，卻分別在每行與每列之間比較。某張圖對許多文字都有高分，可能在行內勝出，但對某段文字而言另一個圖片分數更高。我們可以讓兩個方向都提供學習信號，而不只顧其中一種查詢。

前置是[10.10把配對當分類](https://birdhackor.github.io/tiny-perceptron-vlm/10.10.html)與[W.5矩陣轉置](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.5)。相似度矩陣S原本行是圖、列是文字，轉置S.T以後行變文字、列變圖。兩邊正確配對都在索引0、1，所以可用相同標籤。雙向代價是兩種交叉熵的平均，不需要改變向量寬度。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch.nn import functional as F

scores = torch.tensor([[2.0, 0.1], [0.4, 1.8]], requires_grad=True)
labels = torch.arange(2)
image_to_text = F.cross_entropy(scores, labels)
text_to_image = F.cross_entropy(scores.T, labels)
loss = (image_to_text + text_to_image) / 2
loss.backward()
print("圖找文", round(image_to_text.item(), 4))
print("文找圖", round(text_to_image.item(), 4))
print("平均", round(loss.item(), 4))
print("梯度符號", torch.sign(scores.grad).tolist())

輸入四個手設分數，矩陣不對稱：圖0對文字1為0.1，圖1對文字0為0.4。因此轉置之後各行比較的差距確實不同。第一方向輸出約0.1799，第二方向約0.1758，平均約0.1779；數值可在執行時核對。最後梯度符號是`[[-1,1],[1,-1]]`，兩種比較合起來仍鼓勵這裡的正確對角配對。

兩個方向共用同一矩陣，所以一個元素的梯度包含圖查詢與文字查詢兩種影響。只加其中一種代價並不讓資料形狀錯誤，但訓練目標有所偏重。真檢索評估也應分別拿圖找文字、拿文字找圖，檢查各自是否選中正確身份；相同描述對應多圖時，仍需要處理上一節的多個正確配對問題。

檢索成功不等於會生成描述：選卡片只需比較有限候選，寫句子則要逐步預測許多文字。兩者可以共享特徵，但任務證據要分別報告。

第一行圖找文比較2.0與0.1，第二行比較0.4與1.8；文找圖則比較第一列2.0與0.4、第二列0.1與1.8。這四組候選差距分別1.9、1.4、1.6、1.7，因此兩個平均代價接近卻不必相等。如果矩陣完全對稱，轉置看起來一樣，容易讓人誤以為雙向只是重複計算；本例刻意不對稱，讓差異可追蹤。也正因為每列各有自己的候選競爭，某一方向表現好，不會在數學上自動保證另一方向每個查詢都好。

練習把平均改成`(2*image_to_text + text_to_image)/3`，先預測形狀不變、總分更靠近圖找文代價，再執行核對。權重改變表達你對兩個方向的重視程度，不是增加了新的圖片資訊。
